# 06 · Bonus: bootstrap a golden dataset with synthetic test generation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/ragas-evals-enterprise/blob/main/notebooks/06_bonus_synthetic_testset.ipynb)

**Goal:** hand-writing hundreds of golden questions is slow. Ragas can **generate candidate test cases** from
your documents, using LangChain models directly, which you then **review** and promote into the golden set.

| | |
|---|---|
| ⏱️ Time | ~15 minutes |
| 💵 API cost | ~$0.02 |

### 🔑 Before you run anything
* **Colab:** click the 🔑 **Secrets** icon in the left sidebar → *Add new secret* → name `OPENAI_API_KEY`,
  paste your key, and switch on **Notebook access**. (If you skip this, the setup cell asks you to paste the key.)
* **Local:** copy `.env.example` to `.env` and put your key there. See the README for the full setup.

The setup cell clones the project, installs the pinned requirements (~1 min on Colab), and loads your key.
If Colab prints pip *dependency-resolver warnings* about other preinstalled packages, they are harmless.

In [1]:
# ⚙️ SETUP: run this cell first. It works in Google Colab and on your laptop.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/baluragala/ragas-evals-enterprise.git"  # ← change this if you forked the repo

if "google.colab" in sys.modules:                       # running in Colab: fetch the project + dependencies
    if not Path("/content/ragas-evals-enterprise").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ragas-evals-enterprise"], check=True)
    os.chdir("/content/ragas-evals-enterprise")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Find the project root (works from notebooks/ or the repo root) and make `aurora_evals` importable.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "aurora_evals").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["RAGAS_DO_NOT_TRACK"] = "true"   # turn off ragas telemetry (it also makes evals ~5x faster; see docs)

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)

from aurora_evals.config import setup_environment
setup_environment()   # finds OPENAI_API_KEY in your env / .env / Colab Secrets, or asks you to paste it
print("✅ Ready. Project root:", ROOT)

✅ Ready. Project root: /Users/balakrishna/Training/gen-ai/ragas-evals-enterprise


## 1. How the generator works

`TestsetGenerator` builds a **knowledge graph** from your documents (it extracts headlines, summaries, entities
and themes, then links related chunks), then synthesises questions in several styles:

* **single-hop specific**: one fact from one chunk
* **multi-hop abstract / specific**: questions that need two connected chunks

It also invents **personas** (e.g. "a frequent traveller", "a parent opening a teen account") and query styles,
including **misspellings and casual phrasing**, to imitate real users.

`TestsetGenerator.from_langchain(...)` accepts LangChain chat and embedding models directly.

> ℹ️ This runs fine in notebooks. In a plain Python script, call it inside an already-running event loop (e.g.
> inside `async def main()` run by `asyncio.run`). Otherwise LangChain's async HTTP client can fail with
> `Event loop is closed` (see docs/04_troubleshooting.md).

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from aurora_evals.knowledge_base import load_documents

generator = TestsetGenerator.from_langchain(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    embedding_model=OpenAIEmbeddings(model="text-embedding-3-small"),
)
testset = generator.generate_with_langchain_docs(load_documents(), testset_size=8)
synthetic = testset.to_pandas()
synthetic[["user_input", "reference", "synthesizer_name"]]

,user_input,reference,synthesizer_name
0,What are the requirements and procedures for opening an Aurora Teen Checking account for minors?,"Teens aged 13–17 can open an Aurora Teen Checking account, which must be done jointly with a parent or legal guardian who is an existing Aurora customer. Th...",single_hop_specific_query_synthesizer
1,What is the significance of the 60 days period in the context of disputing transactions and zero-liability protection?,The 60 days period is significant because you are not responsible for unauthorized transactions if you report them within 60 days of the statement date on w...,single_hop_specific_query_synthesizer
2,What kinda benefits Visa cards from Aurora Bank got?,"Aurora Bank offers three consumer credit cards issued on the Visa network, which include contactless payments, virtual card numbers in the Aurora mobile app...",single_hop_specific_query_synthesizer
3,Wut r the requirements for opening a savings account and how does the APY vary based on balance?,"To open a savings account, U.S. citizens and permanent residents aged 18 or older must provide a valid, unexpired government-issued photo ID, proof of addre...",multi_hop_abstract_query_synthesizer
4,What steps should a customer take to dispute a transaction and how does zero-liability fraud protection apply to unauthorized transactions?,"To dispute a transaction, a customer can do so for up to 60 days after the statement date on which it appears. They can open a dispute in the app by selecti...",multi_hop_abstract_query_synthesizer
5,"What are the benefits of having an Aurora Platinum card when making wire transfers, and what are the fees associated with domestic and international wires?","The Aurora Platinum card offers significant benefits for wire transfers, particularly for cardholders who also maintain an Aurora checking account. Specific...",multi_hop_abstract_query_synthesizer
6,"What are the key features of Aurora Bank's savings accounts and credit cards, and how do they cater to different financial needs?","Aurora Bank offers a variety of financial products tailored to different needs. For savings accounts, they provide a high-yield savings account with an Annu...",multi_hop_specific_query_synthesizer
7,What are the benefits of the Aurora Platinum credit card compared to the wire transfer benefits for cardholders who also hold an Aurora checking account?,"The Aurora Platinum credit card offers several benefits, including an annual fee of $450, a purchase APR of 20.99% variable, and rewards of 5x points on fli...",multi_hop_specific_query_synthesizer
8,What data does Aurora Wealth collect and how long do they keep it?,"Aurora Wealth collects identity information, account and transaction data, device information from the mobile app, and records of your communications with t...",multi_hop_specific_query_synthesizer


## 2. Review before you trust

Synthetic questions are **candidates**, not gold. Review each one with a subject-matter expert:

| check | why |
|---|---|
| Would a real customer ask this? | generators produce oddly formal or trivial questions |
| Is the `reference` actually correct and complete? | the generator's LLM can be wrong too |
| Does it duplicate an existing case? | duplicates inflate scores |
| Which tags apply (`question_type`, `category`)? | needed for slicing |
| What's missing? | generators rarely produce **adversarial** or **unanswerable** cases; write those by hand |

Below we save the candidates in the golden format with `"status": "needs_review"`, in a **separate** file. They
are never merged automatically.

In [3]:
import json
from pathlib import Path

candidates = [
    {"id": f"syn-{i:03d}", "user_input": r.user_input, "reference": r.reference, "reference_sources": [],
     "category": "synthetic", "question_type": r.synthesizer_name, "difficulty": "unknown", "status": "needs_review"}
    for i, r in enumerate(synthetic.itertuples(), start=1)
]
out = ROOT / "data" / "golden" / "synthetic_candidates.jsonl"
out.write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in candidates) + "\n")
print(f"wrote {len(candidates)} candidates → {out.relative_to(ROOT)}")

wrote 9 candidates → data/golden/synthetic_candidates.jsonl


## 3. A first signal: run the assistant on the synthetic set

Even before review, synthetic cases are useful for a quick **robustness** check: typos and unusual phrasings
stress retrieval.

In [4]:
from aurora_evals.evaluation import evaluate_rag
from aurora_evals.metrics import rag_quality_suite

metrics = [m for m in rag_quality_suite() if m.name in ("context_recall", "faithfulness", "factual_correctness")]
syn_run = await evaluate_rag("baseline", metrics, samples=candidates, run_name="baseline_on_synthetic")
syn_run.df[["id", "user_input", "context_recall", "faithfulness", "factual_correctness"]]

,id,user_input,context_recall,faithfulness,factual_correctness
0,syn-001,What are the requirements and procedures for opening an Aurora Teen Checking account for minors?,0.666667,1.000000,0.89
1,syn-002,What is the significance of the 60 days period in the context of disputing transactions and zero-liability protection?,1.000000,1.000000,0.75
2,syn-003,What kinda benefits Visa cards from Aurora Bank got?,1.000000,1.000000,1.00
3,syn-004,Wut r the requirements for opening a savings account and how does the APY vary based on balance?,0.666667,1.000000,0.75
4,syn-005,What steps should a customer take to dispute a transaction and how does zero-liability fraud protection apply to unauthorized transactions?,1.000000,1.000000,0.89
5,syn-006,"What are the benefits of having an Aurora Platinum card when making wire transfers, and what are the fees associated with domestic and international wires?",0.400000,0.800000,0.31
6,syn-007,"What are the key features of Aurora Bank's savings accounts and credit cards, and how do they cater to different financial needs?",0.444444,0.666667,0.39
7,syn-008,What are the benefits of the Aurora Platinum credit card compared to the wire transfer benefits for cardholders who also hold an Aurora checking account?,0.750000,1.000000,0.78
8,syn-009,What data does Aurora Wealth collect and how long do they keep it?,1.000000,0.000000,0.00


## ✅ Key takeaways
1. Synthetic generation **accelerates** building a golden set; it does not replace expert review.
2. Write **adversarial and unanswerable** cases by hand. Generators under-produce exactly the cases that hurt most.
3. Keep candidates **separate** with a review status; merge deliberately (and re-baseline afterwards).

🎓 **You've finished the course!** Revisit `docs/03_enterprise_playbook.md` for the checklist of practices you
can take to a real project.